> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Consumption Intelligence & Utilization Quadrants**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../06_consumption_intelligence.ipynb`.

# Notebook 06: Consumption Intelligence & Utilization Quadrants

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 06 of 12  
**Focus:** Multidimensional Telemetry, Correlation Matrices, Cost vs Utilization Quadrants, Unit Economics Curves  

---

## 1. Business Problem

A fundamental flaw in legacy IT financial management is evaluating costs in isolation from consumption telemetry. When an application's annual expenditure increases by 25%, finance teams instinctively flag it as a cost overrun. However:
- If transactions grew by 40%, the application's **unit economics actually improved**.
- Conversely, if an application's spend stayed flat while active user volume declined by 70%, its unit cost escalated dramatically.

To separate genuine operational scaling from technical waste, we must fuse financial ledgers with **operational consumption telemetry**.

---

## 2. Core Concept & Formulation

### 2.1 Multidimensional Telemetry Vector
For each application $i$ in month $m$, we track:
$$\mathbf{C}_{i,m} = [\text{Active Users}, \text{Transactions}, \text{API Calls}, \text{Compute Hours}, \text{Storage GB}, \text{Service Tickets}]$$

### 2.2 Objective Median-Split Quadrant Matrix
Rather than using arbitrary subjective thresholds, we perform median-split clustering across the entire portfolio:
- $\text{Median Cost} = \text{Median}(\{\text{TCO}_i\})$
- $\text{Median Users} = \text{Median}(\{\overline{\text{Users}}_i\})$

| Quadrant | Economic Signature | Strategic Business Interpretation |
| :--- | :--- | :--- |
| **High Cost / High Usage** | Scale Anchor | Core operational engines. Focus on architectural stability and continuous FinOps. |
| **High Cost / Low Usage** | Investigate / Rationalize | Heavy financial burden with minimal adoption. High-priority rationalization review. |
| **Low Cost / High Usage** | High Efficiency Workhorse | High organizational value with low overhead. Models for engineering best practices. |
| **Low Cost / Low Usage** | Niche Utility | Fit-for-purpose specialized utilities. Low financial impact; leave alone unless legacy risk. |

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config
from tvi.consumption_analytics import analyze_consumption_quadrants, find_high_cost_low_utilization, analyze_unit_economics_trends
from tvi.database import get_database

config = load_config()
db = get_database(config.database.path)

# Classify portfolio into objective consumption quadrants
quadrant_df = analyze_consumption_quadrants()
print("Consumption Quadrant Analysis Compiled:")
display(quadrant_df[["application_id", "application_name", "annual_tco", "avg_monthly_users", "consumption_quadrant"]].head(8))

Consumption Quadrant Analysis Compiled:


---

## 3. Multivariate Telemetry Correlation Heatmap

We examine correlations between all 6 telemetry vectors across the application portfolio:

In [2]:
# Compute Correlation Matrix across Telemetry Vectors
telemetry_summary = db.query_df("""
    SELECT
        application_id,
        AVG(total_active_users) as users,
        SUM(total_transactions) as transactions,
        SUM(total_api_calls) as api_calls,
        SUM(total_compute_hours) as compute_hours,
        AVG(total_storage_gb) as storage_gb,
        SUM(total_tickets) as tickets
    FROM v_app_monthly_consumption
    GROUP BY application_id
""")

corr = telemetry_summary.drop(columns=["application_id"]).corr()

plt.figure(figsize=(7, 5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Pearson Correlation")
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr)), corr.columns)
plt.title("Correlation Matrix: Operational Telemetry Vectors", fontsize=12)

for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black" if abs(corr.iloc[i, j]) < 0.6 else "white", fontsize=9)

plt.tight_layout()
plt.show()

---

## 4. Interactive 2D Bubble Quadrant Matrix (Plotly)

We render an interactive Plotly bubble chart showing the 4 quadrants with median threshold lines and hover inspection:

In [3]:
med_cost = quadrant_df["annual_tco"].median()
med_users = quadrant_df["avg_monthly_users"].median()

fig = px.scatter(
    quadrant_df,
    x="avg_monthly_users",
    y="annual_tco",
    color="consumption_quadrant",
    size="annual_cost_per_user",
    hover_name="application_name",
    hover_data={"application_id": True, "annual_tco": ":,.0f", "avg_monthly_users": ":.0f", "consumption_quadrant": True},
    title="Technology Portfolio: Cost vs Utilization Quadrant Matrix",
    labels={"avg_monthly_users": "Active Users", "annual_tco": "Annual TCO (₹)"},
    template="plotly_white",
    color_discrete_map={
        "High Cost / High Utilization (Scale Anchor)": "#1f77b4",
        "High Cost / Low Utilization (Investigate for Rationalization)": "#d62728",
        "Low Cost / High Utilization (High Efficiency Workhorse)": "#2ca02c",
        "Low Cost / Low Utilization (Niche Utility)": "#7f7f7f"
    }
)

fig.add_hline(y=med_cost, line_dash="dash", line_color="grey", annotation_text=f"Median Spend (₹{med_cost/1e7:.1f} Cr)")
fig.add_vline(x=med_users, line_dash="dot", line_color="grey", annotation_text=f"Median Users ({med_users:.0f})")
fig.update_layout(yaxis=dict(tickformat="~s"))
fig.show()

---

## 5. 12-Month Unit Cost Trajectory Time Series

We analyze monthly unit cost trajectories (Cost per Digital Transaction) over the 12 months of FY2024, contrasting:
- `APP001` (Scale Anchor)
- `APP014` (Scenario G: Scaling volume)
- `APP022` (Scenario H: Cost anomaly)

In [4]:
unit_trends = analyze_unit_economics_trends()

sample_apps = ["APP001", "APP014", "APP022"]
plt.figure(figsize=(10, 4.5))

for a_id in sample_apps:
    sub = unit_trends[unit_trends["application_id"] == a_id]
    plt.plot(sub["month"], sub["monthly_cost_per_transaction"], marker="o", label=f"{a_id} ({sub.iloc[0]['application_name']})", linewidth=2)

plt.title("Monthly Cost per Transaction Trajectory (FY2024)", fontsize=13)
plt.ylabel("Cost per Transaction (₹)", fontsize=11)
plt.xlabel("Month", fontsize=11)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---

## 6. Active Development (v2): GreenOps, Decomposition & Rightsizing

In the **v2 Architecture**, consumption intelligence extends to sustainability and automated optimization:
1. **Time-Series Telemetry STL Decomposition:** Decomposes consumption into Trend, Seasonal, and Residual signals.
2. **Serverless Concurrency & Auto-Scaling Telemetry:** Models peak executions, cold starts, and scaling efficiency.
3. **GreenOps Environmental Footprint:** Computes energy usage (kWh) and greenhouse gas emissions (kg CO2e) per application.
4. **Automated Infrastructure Rightsizing:** Identifies over-provisioned capacity with projected monthly and annual savings.

In [5]:
from tvi.consumption_analytics import (
    decompose_telemetry_time_series,
    generate_serverless_concurrency_metrics,
    calculate_greenops_carbon_footprint,
    generate_rightsizing_recommendations,
)

# 1. Telemetry Trend & Seasonal Decomposition (APP014 Elastic Surge)
decomp = decompose_telemetry_time_series("APP014", metric="total_transactions")
print(f"Time-Series Decomposition for APP014: Trend Slope = {decomp['trend_slope']:.2f}, Residual Std = {decomp['residual_std']:.2f}")
decomp_df = pd.DataFrame({
    "Month": decomp["months"],
    "Actual Txns": decomp["actual"],
    "Trend": decomp["trend"],
    "Seasonal": decomp["seasonal"],
    "Residual": decomp["residual"]
})
display(decomp_df.head(6))

# 2. Serverless Execution & Auto-Scaling Concurrency
serverless_df = generate_serverless_concurrency_metrics()
print("\nServerless Concurrency & Auto-Scaling Elasticity Profile:")
display(serverless_df.head(5))

# 3. GreenOps Environmental Footprint (CO2e Emissions)
greenops_df = calculate_greenops_carbon_footprint()
print(f"\nGreenOps Sustainability Footprint (Top Carbon Emitting Applications):")
print(f"Total Portfolio Emissions: {greenops_df['emissions_metric_tons'].sum():.2f} Metric Tons CO2e")
display(greenops_df[["application_id", "application_name", "annual_energy_kwh", "annual_emissions_kg_co2e", "emissions_metric_tons"]].head(5))

# 4. Automated Infrastructure Rightsizing Recommendations
rightsizing_df = generate_rightsizing_recommendations()
total_rightsize_savings = rightsizing_df["projected_annual_savings_inr"].sum() / 1e7
print(f"\nAutomated Rightsizing Opportunities: {len(rightsizing_df)} Recommendations (Total Savings: ₹{total_rightsize_savings:.2f} Cr/yr)")
display(rightsizing_df[["application_id", "application_name", "recommendation_action", "implementation_complexity", "projected_monthly_savings_inr", "projected_annual_savings_inr"]].head(6))

Time-Series Decomposition for APP014: Trend Slope = 362440.71, Residual Std = 712229.01

Serverless Concurrency & Auto-Scaling Elasticity Profile:

GreenOps Sustainability Footprint (Top Carbon Emitting Applications):
Total Portfolio Emissions: 94.33 Metric Tons CO2e

Automated Rightsizing Opportunities: 6 Recommendations (Total Savings: ₹1.88 Cr/yr)


## 6. Limitations & Methodological Guardrails

1. **Labeling Guardrail:** We intentionally avoid labeling quadrants as "good" or "bad". A low-user application might be an executive risk engine used once a quarter to make multi-billion rupee capital allocation decisions.
2. **Investigation Mandate:** The quadrant matrix generates an **indicator for human investigation**, not an automated termination order.

## 7. Next Step

In **Notebook 07: Application Rationalization**, we formalize a multi-criteria decision index (RRI) that combines cost, utilization, capability redundancy, criticality, and lifecycle status to systematically prioritize portfolio rationalization.